# Random forest models for biome prediction

Classify observed biomes and estimate LPJ-GUESS vegetation carbon and net primary productivity from climate and soil predictors. See the [README](README.md) for setup and data access.


In [ ]:

#Project 2
import pandas as pd
import cartopy.crs as ccrs # Enables geographic mapping, e.g., drawing coastlines, etc
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.model_selection import cross_val_score, train_test_split, KFold, GridSearchCV
from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score, classification_report
import shap as shap_library
import seaborn as sns
import numpy as np
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.metrics import mean_squared_error


proj = ccrs.PlateCarree()

In [ ]:
from pathlib import Path
import os

data_dir = Path(os.environ.get("BIOME_DATA_DIR", "data"))
Tswrfdaymean1961_1990_stats = data_dir / "Tswrfdaymean1961_1990_stats.csv"
Tmpdaymean1961_1990_stats = data_dir / "Tmpdaymean1961_1990_stats.csv"
Tmindaymean1961_1990_stats = data_dir / "Tmindaymean1961_1990_stats.csv"
Tmaxdaymean1961_1990_stats = data_dir / "Tmaxdaymean1961_1990_stats.csv"
Predaymean1961_1990_stats = data_dir / "Predaymean1961_1990_stats.csv"
LPJGUESS_output_BERN1 = data_dir / "LPJ-GUESS_output_BERN1.csv"
soilmap = data_dir / "soilmap.dat"
if not soilmap.is_file():
    soilmap = data_dir / "soilmap2.dat"
biome_legend = data_dir / "legend of biomes.txt"
if not biome_legend.is_file():
    biome_legend = data_dir / "legend-of-biomes.txt"
gridlist = data_dir / "gridlist_pan_gfed_ISO3_UN.txt"

required = (
    Tswrfdaymean1961_1990_stats, Tmpdaymean1961_1990_stats,
    Tmindaymean1961_1990_stats, Tmaxdaymean1961_1990_stats,
    Predaymean1961_1990_stats, LPJGUESS_output_BERN1,
    soilmap, biome_legend, gridlist,
)
missing = [path.name for path in required if not path.is_file()]
if missing:
    raise FileNotFoundError(f"Missing files in {data_dir.resolve()}: {', '.join(missing)}. See README.md.")


In [ ]:
def format_rbg_string(rgbstring):
    """Transform a RGB string to a tuple of floats that matplotlib can understand"""
    rgbs = rgbstring.split(' ')
    rgbtup = tuple(float(rgb) for rgb in rgbs)
    return rgbtup

with open(biome_legend, 'r') as f:
    lines = [ l.strip() for l in f.readlines()] # Remove endline characters and spaces

biome_names = lines[slice(0, len(lines), 2)] # Only use the biome names, not the colors
biome_names = [ ' '.join(l.split(' ')[1:]) for l in biome_names ] # Remove number in front of names
biome_rgbs = lines[slice(1, len(lines), 2)] # Filter out biome RGB colors
biome_rgbs = [format_rbg_string(rgbstr) for rgbstr in biome_rgbs]
biome_names

In [ ]:
countries = pd.read_csv(gridlist, sep='\\s+')
countries.head()

In [ ]:
# Explore the options for region/country
countries['Pan_2007'].value_counts()

In [ ]:
# Biome obs from Haxeltine & Prentice, 1996
df = pd.read_csv(LPJGUESS_output_BERN1)
df.head()

In [ ]:
biome_data = df[['Lon', 'Lat', 'Biome_obs']]
biome_data['Biome_obs'].value_counts()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5), subplot_kw=dict(projection=proj))

# Hacky way of creating a map, pretent it is a scatterplot with lons and lats on x and y!
for i, name in enumerate(biome_names):
    biome = biome_data[biome_data['Biome_obs'] == i +1]
    biome.plot(x='Lon', y='Lat', color=biome_rgbs[i], ax=ax, kind='scatter', s=.6, label=name, marker=',')


ax.legend(markerscale=10, loc='center right', bbox_to_anchor=(1.27, .5)) # Make markers bigger
ax.coastlines()
plt.tight_layout()
plt.show()

# **PART 1: Data**

Data formatting below

In [ ]:
TswrfDF = pd.read_csv(Tswrfdaymean1961_1990_stats)
TmpmeanDF = pd.read_csv(Tmpdaymean1961_1990_stats)
TminDF = pd.read_csv(Tmindaymean1961_1990_stats)
TmaxDF = pd.read_csv(Tmaxdaymean1961_1990_stats)
PreDF = pd.read_csv(Predaymean1961_1990_stats)
LPJGuessDF = pd.read_csv(LPJGUESS_output_BERN1)

In [ ]:
# Shapes
print(TswrfDF.shape)
print(TmpmeanDF.shape)
print(TminDF.shape)
print(TmaxDF.shape)
print(PreDF.shape)
print(LPJGuessDF.shape)

In [ ]:
# Reading .dat file
soilDF = pd.read_csv(soilmap, sep='\t')
soilDF = soilDF.rename(columns={'lon':'Lon', 'lat':'Lat'})
soilDF.head()

In [ ]:
for column in soilDF.columns.to_list():
    soilDF[column] = pd.to_numeric(soilDF[column], errors='coerce')

In [ ]:
# Shape
print(soilDF.shape)

In [ ]:
soilDF.head()

In [ ]:
# Drop lat and lon from all but tswrfDF
TmpmeanDF = TmpmeanDF.drop(columns=['Lon', 'Lat'])
TminDF = TminDF.drop(columns=['Lon', 'Lat'])
TmaxDF = TmaxDF.drop(columns=['Lon', 'Lat'])
PreDF = PreDF.drop(columns=['Lon', 'Lat'])
LPJGuessDF = LPJGuessDF.drop(columns=['Lon', 'Lat'])

In [ ]:
# Dictionary for column names to new names
rename_tswrf = {oldname:"Tswrf" + oldname for oldname in TswrfDF.columns.to_list()[2:]}
rename_tmp = {oldname:"Tmp" + oldname for oldname in TmpmeanDF.columns.to_list()}
rename_tmin = {oldname:"Tmin" + oldname for oldname in TminDF.columns.to_list()}
rename_tmax = {oldname:"Tmax" + oldname for oldname in TmaxDF.columns.to_list()}
rename_pre = {oldname:"Pre" + oldname for oldname in PreDF.columns.to_list()}

# Rename the columns
TswrfDF = TswrfDF.rename(columns=rename_tswrf)
TmpmeanDF = TmpmeanDF.rename(columns=rename_tmp)
TminDF = TminDF.rename(columns=rename_tmin)
TmaxDF = TmaxDF.rename(columns=rename_tmax)
PreDF = PreDF.rename(columns=rename_pre)

In [ ]:
# Join dataframes together, except for LPJGuess values
data_df = TswrfDF.join([TmpmeanDF, TminDF, TmaxDF, PreDF])

In [ ]:
# Merge with soilDF, in proper order
data_df = data_df.merge(soilDF, on=['Lon', 'Lat'], how='left')
data_df.head()

In [ ]:
# Target variable (biome classification) is Biome_obs in the LPJGuess df
biome_obs = LPJGuessDF['Biome_obs']
biome_obs[:5]

In [ ]:
# What is the spread of biomes?
print(biome_obs.value_counts())
# and the unique values?
print(set(biome_obs.values))
plt.hist(biome_obs.values, bins=len(set(biome_obs.values)))
plt.title("Observed biomes")
plt.show()

In [ ]:
# Drop the observed target variable from the dataframe
# Unnecessary unless LPJ guess was merged into the df
#data_df = data_df.drop(columns=['Biome_obs'])
#data_df.head()

In [ ]:
# Check for nan values
num_nan = data_df.isnull().values.sum()
print(f"{num_nan} nan values found")
if num_nan > 0:
    print(f"at: {np.where(data_df.isnull().values)}")
    nan_df = data_df.iloc[data_df.isnull().values]
    nan_df.head()

In [ ]:
#CallData


#Lon,Lat,SpringMean,SpringMedian,SpringStd,SummerMean,SummerMedian,SummerStd,FallMean,FallMedian,FallStd,WinterMean,WinterMedian,WinterStd
def CallData(obsIndex=float('nan'), parameter=float('nan'), statParameter=float('nan'), all=False):

  data = data_df

  parameterDict = {'Tswrf': 0, #radiation
                    'Pre': 1,
                    'Tmp': 2,
                    'Tmax': 3,
                    'Tmin': 4,
                    'lpjGuess': 5,
                    'soildata': 6,} #file

  StatDict = {'Lon': 0,
                   'Lat': 1, #the first two only applies to the first collumn Tswrf
                   'SpringMean': 2,
                   'SpringMedian': 3,
                   'SpringStd': 4,
                   'SummerMean': 5,
                   'SummerMedian': 6,
                   'SummerStd': 7,
                   'FallMean':8,
                   'FallMedian': 9,
                   'FallStd':10,
                   'WinterMean':11,
                   'WinterMedian':12,
                   'WinterStd':13, #conisders the 14 size datas
                    'NPP': 0, #applies only to the last 9 row values
                    'SoilR': 1,
                    'MaxBiomeCmass': 2,
                    'MxbiomeLAI':3,
                    'VegC':4,
                    'LitterC':5,
                    'SoilC':6,
                    'Biome_Cmass':7,
                    'Biome_LAI':9,
                    'clay': 0, #last soildata
                  	'silt':1,
                  	'sand':2,
                  	'orgC':3,
                  	'CN':4,
                  	'pH':5,
                  	'cellfraction':6


              }
              #12*5 +2 collumns
  paramIdx = parameterDict[parameter]
  statIdx = StatDict[statParameter]


  if all == True and obsIndex != obsIndex== float('nan'):
    return data.iloc[obsIndex, :]
  elif paramIdx != float('nan')  and obsIndex != float('nan') and statIdx != float('nan'):
    #paramter will give us how many collumn groups we have to move
    collumnList = []
    Tswrfcollumn = data.iloc[obsIndex,0:14] #14 elements
    Precollumn = data.iloc[obsIndex,14:26] #12 elements
    Tmpcollumn = data.iloc[obsIndex,26:38]
    Tmaxcollumn = data.iloc[obsIndex,38:50]
    Tmincollumn = data.iloc[obsIndex,50:62]
    LpjGuesscollumn = data.iloc[obsIndex,62:71] #9elements
    soildatacollumn = data.iloc[obsIndex,71:78] #7 elements

    #print(Tswrfcollumn)

    collumnList.append(Tswrfcollumn)
    collumnList.append(Precollumn)
    collumnList.append(Tmpcollumn)
    collumnList.append(Tmaxcollumn)
    collumnList.append(Tmincollumn)
    collumnList.append(LpjGuesscollumn)
    collumnList.append(soildatacollumn)

    return collumnList[paramIdx].iloc[statIdx]

  else:
    return float('nan')






alpha = CallData(obsIndex=4, parameter='Tswrf', statParameter='WinterMean', all=False)

print(alpha)




















# **Part 2: Data Description and Visualization**

 **WE NEED TO CREATE A PANDAS DATAFRAME WITHOUT THE LPJ GUESS OUTPUT VARIABLES AND USE THIS FOR THE CODE BELOW**

In [ ]:
#NaN handlening: REPLACING NAN VARIABLES WITH COLLUMN MEAN
#checking for NaNs

#train_df = data_df.drop(data_df.columns[62:71], axis=1) #train data, dropping every collumn related to lpj guess
train_df = data_df.copy() # LPJ Guess values haven't been loaded into the frame

num_cols = train_df.select_dtypes(include='number').columns

#Check for NaNs before
print("Any NaNs left anywhere? ->", train_df.isna().any().any())
print("Columns still with NaNs:\n", train_df.columns[train_df.isna().any()])
print("NaN counts (numeric cols):\n", train_df[num_cols].isna().sum())


#  Replace inf/-inf with NaN (these also break VIF)
train_df[num_cols] = train_df[num_cols].replace([np.inf, -np.inf], np.nan)

# Fill NaNs with each column's mean
train_df[num_cols] = train_df[num_cols].apply(lambda s: s.fillna(s.mean()))

#check for NaNs

print("Any NaNs left anywhere? ->", train_df.isna().any().any())
print("Columns still with NaNs:\n", train_df.columns[train_df.isna().any()])
print("NaN counts (numeric cols):\n", train_df[num_cols].isna().sum())


In [ ]:
#VIF VERIFICATION: checking for colinearity [THIS takes a couple of minutes to run]

# Compute VIF for each feature
vif_data = pd.DataFrame()
vif_data['Feature'] = train_df.columns
vif_data['VIF'] = [variance_inflation_factor(train_df.values, i)
                   for i in range(train_df.shape[1])]

print(vif_data)

In [ ]:
#Rule of thumb if VIF > 10 high colinearity


vif_filteredData = filtered = vif_data[vif_data['VIF'] < 20]



print(len(vif_filteredData))

#vif_filteredData

#we do a correlation matrix for the filtered VIF Data


NUMERIC_COLUMNS = vif_filteredData['Feature'].astype(str).tolist()
correlation_matrix = train_df[NUMERIC_COLUMNS].corr()

plt.figure(figsize=(8, 6))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5, vmin=-1, vmax=1)

plt.title('Correlation Matrix Heatmap')
plt.show()



In [ ]:
#Training data (after VIF) (Re run this after editing training matrix)

train_filtered = train_df[NUMERIC_COLUMNS]

#sns.pairplot(train_filtered)




## Grid search and evaluation functions for classifiers and regression models

In [ ]:
# Function to run grid search and return the best estimator
def grid_search(x_train, y_train, n_depths_range, n_estimators_range, min_leaf_samples_range, k_folds, classifier, score = "accuracy"):
    kfold5 = KFold(k_folds, random_state=1, shuffle=True)
    params = {}
    if n_depths_range is not None:
        params['max_depth'] = n_depths_range
    if n_estimators_range is not None:
        params['n_estimators'] = n_estimators_range
    if min_leaf_samples_range is not None:
        params['min_samples_leaf'] = min_leaf_samples_range

    rfc_gscv = GridSearchCV(classifier, param_grid = params, scoring = score,
                                cv = kfold5)
    # Fit the model
    model_rfc = rfc_gscv.fit(x_train, y_train)

    # Model best estimator
    #max_depths=model_rfc.best_estimator_.get_params()["max_depth"]
    #max_trees= model_rfc.best_estimator_.get_params()["n_estimators"]
    #max_cvs= rfc_gscv.best_score_
    best_rf = rfc_gscv.best_estimator_
    print("Best params:", rfc_gscv.best_params_)
    print("CV score:", rfc_gscv.best_score_)
    return best_rf

In [ ]:
# OBSOLETE, was trying something but don't think it's necessary
"""def optimize_params(x_train, y_train, params, best_params, best_score, scoring):
    # Optimize given the parameters in params
    # best_params should hold the best parameters found so far
    params_list = list(params.keys())
    best_params = best_params.copy()
    for key in params_list:
        # Find the best value of this parameter, then optimize the others
        current_best = best_score
        current_best_params = best_params.copy()
        testing_params = current_best_params.copy()
        testing_params[key] *= 1.25
        model = RandomForestClassifier(*testing_params)
        #model.fit(x_train, y_train)
        model_score = cross_val_score(model, x_train, y_train, scoring=scoring)
        if model_score > current_best:
            current_best = model_score
            current_best_params = testing_params
        # Remove the current parameter from the test
        next_params = {k:v for k, v in current_best_params.items() if k != key}
        best_params = optimize_params(x_train, y_train, next_params, current_best_params, current_best, scoring)
    return best_params

# Custom hyperparameter search to check for optimal values
def hyperparameter_search(x_train, y_train, params, scoring="accuracy"):
    # Using the training data, find the optimal RandomForest model by varying the given parameters
    # params should be a dict of key value pairs where each key is a string name of a hyperparameter to optimize
    # and each value is the starting value to begin the search at
    # e.g. {'max_depth': 5, 'n_estimators' : 30, 'min_samples_leaf' : 3}

    # Initialize with given parameters as best score
    model = RandomForestClassifier(*params)
    model_score = cross_val_score(model, x_train, y_train, scoring=scoring)
    return optimize_params(x_train, y_train, params, params, model_score, scoring)"""


In [ ]:
# Model evaluation
def evaluate_model(model, x_train, x_test, y_train, y_test, confusion_matrix=True, calc_shap=True):
    # Analysis of best estimator
    score_rfc_train = accuracy_score(y_train, model.predict(x_train))
    score_rfc_test = accuracy_score(y_test, model.predict(x_test))
    print('Accuracy of RandomForestClassifier on train data: {:.4f}'.format(score_rfc_train))
    print('Accuracy of RandomForestClassifier on test data: {:.4f}'.format(score_rfc_test))

    folds = KFold()

    score_cv_train = cross_val_score(model, x_train, y_train, cv=folds)
    #score_cv_test = cross_val_score(model, x_test, y_test, cv=folds)

    print("Average accuracy of the model on cross validation of train data: %0.4f ( %0.2f)" % (score_cv_train.mean(), score_cv_train.std()))
    #print("Average accuracy of the model on cross validation of test data: %0.4f ( %0.2f)" % (score_cv_test.mean(), score_cv_test.std()))

    print("\nClassification Report:")
    print(classification_report(y_test, model.predict(x_test)))

    if confusion_matrix:
        ConfusionMatrixDisplay.from_estimator(model, x_test, y_test)
        plt.show()

    # Feature importances
    feature_imp = pd.DataFrame(
        {'importance':model.feature_importances_},
        index=x_train.columns)
    print("Feature importances on the best random forest estimator")
    print(feature_imp.sort_values(by='importance', ascending=False))

    # Shap values
    if calc_shap:
        explainer = shap_library.Explainer(model.predict, x_test)
        shap_values = explainer(x_test)
        shap_library.summary_plot(shap_values)

In [ ]:
#regression evaluation
def evaluate_reg(model, x_train, x_test, y_train, y_test, calc_shap = False):
  print(f"The model has a MSE of {mean_squared_error(y_train, model.predict(x_train)):.3f} on the training data")
  print(f"The model has a MSE of {mean_squared_error(y_test, model.predict(x_test)):.3f} on the test data") # we see the blue region extended further as it coveres a larger geographical area with a greater biome diversity, while green points only conisder Russia

  feature_imp = pd.DataFrame(
    {'importance':model.feature_importances_},
    index=x_train.columns)
  print("Feature importances on the best random forest estimator")
  print(feature_imp.sort_values(by='importance', ascending=False))

  # Shap values
  if calc_shap:
      explainer = shap_library.Explainer(model.predict, x_test)
      shap_values = explainer(x_test)
      shap_library.summary_plot(shap_values)


# Part 3: Binary classification

## Regional
We choose data corresponding to two regions from the country data. The training data will be from one region and the test data from the other. The goal is to train a Random Forest binary classifier to distinguish between a certain biome of our choice and all areas not of that biome.

**USE selected_df for the training values as the have been justified for, this dataframe will be used for both train and test and will contain important input variables, country and biome_obs**

In [ ]:
#Nico code

#We need to add country and Biome_observed

#adding countries to the data

countries_added = train_df.merge(countries, on=['Lon', 'Lat'], how='left')

countriesList = countries_added['Pan_2007']

print(countriesList)

train_filtered = train_filtered.join(countriesList, how='left')


#Merging Biome_obs collumn also

train_filtered = train_filtered.join(biome_obs, how = 'left')

print(train_filtered.shape)

train_filtered = train_filtered.join(LPJGuessDF['VegC'], how = 'left')

train_filtered = train_filtered.join(LPJGuessDF['NPP'], how = 'left')

In [ ]:
train_filtered.head()

In [ ]:
# Extract values only from trainCountry and testCountry
train_region = 'Russia'
test_region = 'Canada'

selected_df = train_filtered[train_filtered['Pan_2007'].isin([train_region, test_region])]

#selected_df['Biome_obs']

#We split train and test set according to countries as stated in 3.1 Data split in the PDF

#contains OBS which is our y-values
trainData_y = selected_df[selected_df['Pan_2007'] == train_region]

testData_y = selected_df[selected_df['Pan_2007'] == test_region]



#Y-values for train and test

#WE NEED TO SELECT ONE BIOME, see legend of biomes file

selected_biome = 2 # boreal ever forest
#temp decid forest
#selected_biome =

#we want to define if biome_obs = 5 it is 1 if biome_obs != 5 it is 0


y_train = (trainData_y['Biome_obs'] == selected_biome).astype(int)
y_test  = (testData_y['Biome_obs'] == selected_biome).astype(int)

print(y_train.shape)
print(y_test.shape)


#X-values for train and test
x_train = trainData_y.drop(['Pan_2007', 'Biome_obs', 'Lon', 'cellfraction', 'VegC', 'NPP'], axis=1)

x_test = testData_y.drop(['Pan_2007', 'Biome_obs', 'Lon', 'cellfraction', 'VegC', 'NPP'], axis=1)

In [ ]:
x_train.head()

In [ ]:
# How many of each do we have?
countries['Pan_2007'].value_counts()[np.array([train_region, test_region])]

In [ ]:
# NOTE: Biome_obs contains values from 1-18, so index by subtracting one
print(biome_names)
print(len(biome_names))

In [ ]:
# Look at what biomes are available in this training region
trainData_biome_display = pd.DataFrame({'Biome_obs':trainData_y['Biome_obs'].copy()})
trainData_biome_display['Biome_names'] = np.array([biome_names[val-1] for val in trainData_biome_display['Biome_obs']])
print(trainData_biome_display.value_counts())

In [ ]:
# Look at what biomes are available in this test region
testData_biome_display = pd.DataFrame({'Biome_obs':testData_y['Biome_obs'].copy()})
testData_biome_display['Biome_names'] = np.array([biome_names[val-1] for val in testData_biome_display['Biome_obs']])
print(testData_biome_display.value_counts())

In [ ]:
# How many of each class do we have?
print("Training data counts:")
print(y_train.value_counts())
print("Testing data counts:")
print(y_test.value_counts())

In [ ]:
countries['Pan_2007'].value_counts()

**Fit without k-fold**

In [ ]:
#training the model 1 (no hyper parameter change nor k-fold implementation)

bin_class = RandomForestClassifier(random_state=1)
bin_class.fit(x_train, y_train)

**Fit results without k-fold**

In [ ]:
evaluate_model(bin_class, x_train, x_test, y_train, y_test, confusion_matrix=False, calc_shap=False)

**Fit using k-fold**

In [ ]:
# Use random parameter variation
from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV
from scipy.stats import randint, uniform
#from sklearn.ensemble import RandomForestClassifier

param_dist = {
    "n_estimators": randint(200, 700),          # sample between 200–700
    "max_depth": [6, 10, 16, None],
    "min_samples_leaf": randint(1, 21),         # 1–20
    "max_features": ["sqrt", 0.5],
    "class_weight": [None, "balanced"],
}

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)  # 3-fold is fine to start

rf = RandomForestClassifier(random_state=1, n_jobs=-1)
search = RandomizedSearchCV(
    rf, param_distributions=param_dist,
    n_iter=100,  # try 30–50
    cv=cv, scoring="f1",
    n_jobs=-1, verbose=1, random_state=42
)
search.fit(x_train, y_train)

best_rf = search.best_estimator_
print("Best params:", search.best_params_)
print("CV score:", search.best_score_)
print("Test accuracy:", best_rf.score(x_test, y_test))


In [ ]:
# Using grid search on tree depth and number of estimators to tune hyperparams
k_folds = 5
n_depths_range = np.arange(6, 8, 1, dtype=np.int16)
n_estimators_range = np.arange(30, 150, 30, dtype=np.int16) # CV number of trees (estimators) 10, 20, ..., 100
min_leaf_samples_range = np.arange(3, 6, 3, dtype=np.int16)

best_rf = grid_search(x_train, y_train, n_depths_range, n_estimators_range, min_leaf_samples_range, k_folds,bin_class)

In [ ]:
# Analysis of best estimator
evaluate_model(best_rf, x_train, x_test, y_train, y_test, calc_shap=False)

In [ ]:
explainer = shap_library.Explainer(best_rf.predict, x_test)
shap_values = explainer(x_test)
shap_library.summary_plot(shap_values)

In [ ]:
explainer = shap_library.Explainer(best_rf.predict, x_train)
shap_values = explainer(x_train)
shap_library.summary_plot(shap_values)

In [ ]:
# Try gridsearch without using the pretrained model
k_folds = 5
n_depths_range = np.arange(11, 18, 3, dtype=np.int16)
n_estimators_range = np.arange(100, 170, 30, dtype=np.int16) # CV number of trees (estimators) 10, 20, ..., 100
min_leaf_samples_range = np.arange(1, 6, 2, dtype=np.int16)

best_rf_2 = grid_search(x_train, y_train, n_depths_range, n_estimators_range, min_leaf_samples_range, k_folds, RandomForestClassifier(), score='f1')

Notes on this gridsearch: \
Tried depths from 6-12 in 2s, trees from 60-120 in 30s, and min_samples from 1-16 in 5s. Acc = 0.8632, TP=3009, TN=2601, using depth 12, 120 trees, and 1 min sample.
This suggests allowing higher depth, more trees, and focusing on low sample numbers. \
Tried depths from 11-17 in 3s, trees from 100-160 in 30s, and min_samples from 1-6 in 2s. Acc = 0.8541, TP=3029, TN=2522, using depth 17, 160 trees, and 1 min sample. \
Tried depths from 11-17 in 3s, trees from 100-160 in 30s, and min_samples from 1-6 in 2s. Scored using f1. Acc = 0.8501, TP=3003, TN=2522, using depth 17, 160 trees, and 1 min sample.

In [ ]:
# Analysis of best estimator
evaluate_model(best_rf_2, x_train, x_test, y_train, y_test, calc_shap=False)

## World Data
Here we use all the world data. This gives more samples to train on. We use biome 2, which is boreal ever forest, as it is the most common.

In [ ]:
# Try training on data from all over the world
selected_df = train_filtered.copy()
trainData_y, testData_y = train_test_split(selected_df, test_size=0.2, random_state=42)

# selected biome
selected_biome = 2 # boreal ever forest

#Y-values for train and test
# Convert to binary variable
y_train = (trainData_y['Biome_obs'] == selected_biome).astype(int)
y_test  = (testData_y['Biome_obs'] == selected_biome).astype(int)

print(y_train.shape)
print(y_test.shape)

#X-values for train and test
x_train = trainData_y.drop(['Pan_2007', 'Biome_obs', 'Lon', 'cellfraction', 'VegC', 'NPP'], axis=1)

x_test = testData_y.drop(['Pan_2007', 'Biome_obs', 'Lon', 'cellfraction', 'VegC', 'NPP'], axis=1)
print(f"Training x shape: {x_train.shape}")
print(f"Test x shape: {x_test.shape}")

In [ ]:
#training the model 1 (no hyper parameter change nor k-fold implementation)

bin_class_world = RandomForestClassifier(random_state=1)
bin_class_world.fit(x_train, y_train)

In [ ]:
evaluate_model(bin_class_world, x_train, x_test, y_train, y_test, confusion_matrix=True, calc_shap=False)

In [ ]:
explainer = shap_library.Explainer(bin_class_world.predict, x_train)
shap_values = explainer(x_train)
shap_library.summary_plot(shap_values)

Notes on binary world model:\
Acc without gridsearch: 0.9693\
With gridsearch: \
Tested with trees from 100-250 in 50s. Acc=0.9688, trees 150

In [ ]:
# Gridsearch over number of trees, since depth and min leaf nodes don't seem to help in our cases
k_folds = 5
n_depths_range = None
n_estimators_range = np.arange(100, 300, 50, dtype=np.int16) # CV number of trees (estimators) 10, 20, ..., 100
min_leaf_samples_range = None

best_binary_rf_world = grid_search(x_train, y_train, n_depths_range, n_estimators_range, min_leaf_samples_range, k_folds, RandomForestClassifier(), score='f1')

In [ ]:
# Analysis of best estimator
evaluate_model(best_binary_rf_world, x_train, x_test, y_train, y_test, calc_shap=False)

In [ ]:
explainer = shap_library.Explainer(best_binary_rf_world.predict, x_train)
shap_values = explainer(x_train)
shap_library.summary_plot(shap_values)

# Part 4: Multiclass Classification


## 4.1 Regional
First, we train a classifier on Russia and Europe and evaluate it on Canada.

In [ ]:

# Extract values only from trainCountry and testCountry
train_region = ['Russia', 'N_Europe']
test_region = ['Canada']

selected_df = train_filtered[train_filtered['Pan_2007'].isin(train_region+test_region)]

#We split train and test set according to countries as stated in 3.1 Data split in the PDF

#contains OBS which is our y-values
trainData_y = selected_df[selected_df['Pan_2007'].isin(train_region)]

testData_y = selected_df[selected_df['Pan_2007'].isin(test_region)]

print(len(trainData_y))

#Y-values for train and test
#we want all biome categories

y_train = trainData_y['Biome_obs']
y_test  = testData_y['Biome_obs']

print(y_train.shape)
print(y_test.shape)

#X-values for train and test
x_train = trainData_y.drop(['Pan_2007', 'Biome_obs', 'Lon', 'cellfraction', 'VegC', 'NPP'], axis=1)

x_test = testData_y.drop(['Pan_2007', 'Biome_obs', 'Lon', 'cellfraction', 'VegC', 'NPP'], axis=1)

In [ ]:
# Biome values in the training region
y_train.value_counts()

In [ ]:
multiclass = RandomForestClassifier(random_state=1)
multiclass.fit(x_train, y_train)

In [ ]:
# Using grid search on tree depth and number of estimators to tune hyperparams
k_folds = 5
n_depths_range = np.arange(6, 8, 1, dtype=np.int16)
n_estimators_range = np.arange(30, 150, 30, dtype=np.int16) # CV number of trees (estimators) 10, 20, ..., 100
min_leaf_samples_range = np.arange(3, 6, 3, dtype=np.int16)

best_rf_multiclass = grid_search(x_train, y_train, n_depths_range, n_estimators_range, min_leaf_samples_range, k_folds,multiclass)

In [ ]:
# Analysis of best estimator
evaluate_model(best_rf_multiclass, x_train, x_test, y_train, y_test, calc_shap=False)

In [ ]:
explainer = shap_library.Explainer(best_rf_multiclass.predict, x_train)
shap_values = explainer(x_train)
shap_library.summary_plot(shap_values)

In [ ]:
# Like in part 3, try gridsearch without the pretrained model
k_folds = 5
n_depths_range = np.arange(6, 13, 2, dtype=np.int16)
n_estimators_range = np.arange(60, 150, 30, dtype=np.int16) # CV number of trees (estimators) 10, 20, ..., 100
min_leaf_samples_range = np.arange(1, 20, 5, dtype=np.int16)

best_rf_multiclass_2 = grid_search(x_train, y_train, n_depths_range, n_estimators_range, min_leaf_samples_range, k_folds, RandomForestClassifier())

In [ ]:
# Analysis of best estimator
evaluate_model(best_rf_multiclass_2, x_train, x_test, y_train, y_test, confusion_matrix=False, calc_shap=False)

## 4.2 Larger Regions
Next, we train a model on more data, picking regions with similar spreads of biomes. We train on North and South America and test on China, South Asia, and Japan.\
Training: Americas, USA \
Testing: China, South_Asia, Japan

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5), subplot_kw=dict(projection=proj))


# Hacky way of creating a map, pretent it is a scatterplot with lons and lats on x and y!
gfed_regions = list(set(countries['GFED-region']))
colors = list(mcolors.CSS4_COLORS)[7:7+len(gfed_regions)]
for i, name in enumerate(gfed_regions):
    gfed_region = countries[countries['GFED-region'] == i +1]
    gfed_region.plot(x='Lon', y='Lat', color=colors[i], ax=ax, kind='scatter', s=.6, label=name, marker=',')


ax.legend(markerscale=10, loc='center right', bbox_to_anchor=(1.27, .5)) # Make markers bigger
ax.coastlines()
plt.tight_layout()
plt.show()

In [ ]:
# Extract values only from desired regions
train_region = ['Americas', 'USA']
test_region = ['China', 'South_Asia', 'Japan']

selected_df = train_filtered[train_filtered['Pan_2007'].isin(train_region+test_region)]

#We split train and test set according to countries as stated in 3.1 Data split in the PDF

#contains OBS which is our y-values
trainData_y = selected_df[selected_df['Pan_2007'].isin(train_region)]

testData_y = selected_df[selected_df['Pan_2007'].isin(test_region)]

print(len(trainData_y))
#Y-values for train and test

#we want all biome categories

y_train = trainData_y['Biome_obs']
y_test  = testData_y['Biome_obs']

print(y_train.shape)
print(y_test.shape)

#X-values for train and test
x_train = trainData_y.drop(['Pan_2007', 'Biome_obs', 'Lon', 'cellfraction', 'VegC', 'NPP'], axis=1)

x_test = testData_y.drop(['Pan_2007', 'Biome_obs', 'Lon', 'cellfraction', 'VegC', 'NPP'], axis=1)

In [ ]:
print(y_train.value_counts())

In [ ]:
print(y_test.value_counts())

In [ ]:
multiclass_americas_china = RandomForestClassifier(random_state=1)
multiclass_americas_china.fit(x_train, y_train)

In [ ]:
evaluate_model(multiclass_americas_china, x_train, x_test, y_train, y_test, confusion_matrix=True, calc_shap=False)

In [ ]:
explainer = shap_library.Explainer(multiclass_americas_china.predict, x_train)
shap_values = explainer(x_train)
shap_library.summary_plot(shap_values)

Notes for training on Americas + USA, tesing on China:
Model with depths 13-19 in 3s, trees 90-150 in 30s, min samples 1-5 in 2s. Acc=0.3123 with 19 depth, 90 trees, 1 min sample \
Suggests more depth might be required \
Model with depths 19-29 in 5s, trees 90-150 in 30s, min samples 1-3 in 2s. Acc: 0.3168 with 19 depth, 120 trees, 1 min sample \
More depth didn't help, might just not be very well balanced.

In [ ]:
# Like in part 3, try gridsearch without the pretrained model
k_folds = 5
n_depths_range = np.arange(19, 30, 5, dtype=np.int16)
n_estimators_range = np.arange(90, 180, 30, dtype=np.int16) # CV number of trees (estimators) 10, 20, ..., 100
min_leaf_samples_range = np.arange(1, 4, 2, dtype=np.int16)

best_rf_americas_china = grid_search(x_train, y_train, n_depths_range, n_estimators_range, min_leaf_samples_range, k_folds, RandomForestClassifier())

In [ ]:
# Analysis of best estimator
evaluate_model(best_rf_americas_china, x_train, x_test, y_train, y_test, confusion_matrix=True, calc_shap=False)

In [ ]:
explainer = shap_library.Explainer(best_rf_americas_china.predict, x_train)
shap_values = explainer(x_train)
shap_library.summary_plot(shap_values)

##4.3 World Data
Here we try training on the data from all over the world.

In [ ]:
lpj_merged_df = TswrfDF[['TswrfSpringStd', 'TswrfSummerStd', 'Lat']].join(LPJGuessDF[['Biome_Cmax', 'Biome_LAI']], how='left')
lpj_merged_df.head()

In [ ]:
# Try training on data from all over the world
# We need the LPJ guess outputs in the same order as our data
selected_df = train_filtered.copy()
# Create merged frame with TswrfSpringStd, TswrfSummerStd, and LPJ biome values
lpj_merged_df = TswrfDF[['TswrfSpringStd', 'TswrfSummerStd', 'Lat']].join(LPJGuessDF[['Biome_Cmax', 'Biome_LAI']], how='left')
selected_df = selected_df.merge(lpj_merged_df, on=['TswrfSpringStd', 'TswrfSummerStd'], how='left')

# Split the data
trainData_y, testData_y = train_test_split(selected_df, test_size=0.2, random_state=42)

print(len(trainData_y))
print(len(testData_y))

# Extract the lpj biome values
lpj_biome_pred_test = testData_y[['Biome_Cmax', 'Biome_LAI']]

#Y-values for train and test

#we want all biome categories

y_train = trainData_y['Biome_obs']
y_test  = testData_y['Biome_obs']

print(y_train.shape)
print(y_test.shape)

#X-values for train and test
x_train = trainData_y.drop(['Pan_2007', 'Biome_obs', 'Lon', 'cellfraction', 'VegC', 'NPP', 'Biome_Cmax', 'Biome_LAI'], axis=1)

x_test = testData_y.drop(['Pan_2007', 'Biome_obs', 'Lon', 'cellfraction', 'VegC', 'NPP', 'Biome_Cmax', 'Biome_LAI'], axis=1)
print(f"Training x shape: {x_train.shape}")
print(f"Test x shape: {x_test.shape}")

In [ ]:
print(y_train.value_counts())

In [ ]:
print(y_test.value_counts())

In [ ]:
multiclass_world = RandomForestClassifier(random_state=1)
multiclass_world.fit(x_train, y_train)

In [ ]:
evaluate_model(multiclass_world, x_train, x_test, y_train, y_test, confusion_matrix=True, calc_shap=False)

In [ ]:
evaluate_model(multiclass_world, x_train, x_test, y_train, y_test, confusion_matrix=True, calc_shap=False)

Notes for training on World data:
Trained with depths 10-30 by 5s, trees 100-200 by 25s, min samples 1-3 by 2s. Acc 0.8345, depth 25, trees 200, min samples 1. \
The default parameters got 0.8361, so we're not doing any better with grid search. Maybe we uncap the depth and min samples and only try varying number of trees?

In [ ]:
# Like in part 3, try gridsearch without the pretrained model
k_folds = 5
n_depths_range = np.arange(10, 35, 5, dtype=np.int16)
n_estimators_range = np.arange(100, 225, 25, dtype=np.int16) # CV number of trees (estimators) 10, 20, ..., 100
min_leaf_samples_range = np.arange(1, 4, 2, dtype=np.int16)

best_rf_world = grid_search(x_train, y_train, n_depths_range, n_estimators_range, min_leaf_samples_range, k_folds, RandomForestClassifier())

In [ ]:
# Analysis of best estimator
evaluate_model(best_rf_world, x_train, x_test, y_train, y_test, confusion_matrix=True, calc_shap=True)

## 4.4: Comparing to LPJGuess
Here we compare the class outputs to those of LPJ Guess in Biome_CMax and Biome_LAi

In [ ]:
# Our best model is multiclass_world, so we use those outputs for comparison
biome_pred = multiclass_world.predict(x_test)
biome_lpj_cmax = lpj_biome_pred_test['Biome_Cmax'].to_numpy()
biome_lpj_lai = lpj_biome_pred_test['Biome_LAI'].to_numpy()

# Plot the values
x_pts = np.arange(1, len(biome_pred)+1)
num = 40
bin_num = len(set(y_train).union(set(y_test)))
x_tick_vals = np.arange(1, bin_num+2)
#plt.scatter(x_pts[:num], biome_pred[:num], label='Our predictions')
#plt.scatter(x_pts[:num], biome_lpj_cmax[:num], label='LPJ Biome_Cmax')
#plt.scatter(x_pts[:num], biome_lpj_lai[:num], label='LPJ Biome_LAI')
plt.hist(biome_pred, bins=bin_num, alpha=0.5, label='Our predictions')
plt.hist(biome_lpj_cmax, bins=bin_num, alpha=0.5, label='LPJ Biome_Cmax')
plt.hist(biome_lpj_lai, bins=bin_num, alpha=0.5, label='LPJ Biome_LAI')
plt.title("Comparing our predictions to LPJ Guess outputs")
plt.xlabel("Predicted Biome")
plt.xticks(x_tick_vals)
plt.ylabel("Count")
plt.legend()
plt.show()

In [ ]:
print(f"LPJ Guess attains an accuracy of {accuracy_score(y_test, biome_lpj_cmax):.4f} on the test data using Cmax")
print(f"LPJ Guess attains an accuracy of {accuracy_score(y_test, biome_lpj_lai):.4f} on the test data using LAI")
print(f"Our best model attains an accuracy of {accuracy_score(y_test, biome_pred):.4f} on the test data")

In [ ]:
print(f"Max biome seen in lpj: {max(np.max(biome_lpj_cmax), np.max(biome_lpj_lai))}")

We see that LPJ guess doesn't do well with these predicted classes, or perhaps the data provided was accidentally scrambled. It doesn't predict any biomes above 13.

# **Part 5: Regression**

## 5.1: VegC
# Using a Random forest regressor to predict the VegC of a place in the world.

### Regional
Here we predict VegC using the Americas, USA, and Canada as training data and Russia as the test region.

In [ ]:
#regression VegC

# Extract values only from desired regions
train_region = ['Americas', 'USA', 'Canada']
test_region = ['Russia']

selected_df = train_filtered[train_filtered['Pan_2007'].isin(train_region+test_region)]

#We split train and test set according to countries as stated in 3.1 Data split in the PDF

#contains OBS which is our y-values
trainData_y = selected_df[selected_df['Pan_2007'].isin(train_region)]

testData_y = selected_df[selected_df['Pan_2007'].isin(test_region)]

print(len(trainData_y))
#Y-values for train and test

#WE NEED TO SELECT ONE BIOME, see legend of biomes file

#selected_biome = 2 # boreal ever forest
#5 temp decid forest

#we want all biome categories

y_train = trainData_y['VegC']
y_test  = testData_y['VegC']

print(y_train.shape)
print(y_test.shape)

#X-values for train and test
x_train = trainData_y.drop(['Pan_2007', 'Biome_obs', 'Lon', 'cellfraction', 'VegC', 'NPP'], axis=1)

x_test = testData_y.drop(['Pan_2007', 'Biome_obs', 'Lon', 'cellfraction', 'VegC', 'NPP'], axis=1)

In [ ]:
vegReg = RandomForestRegressor(random_state=42)

vegReg.fit(x_train, y_train)

In [ ]:
evaluate_reg(vegReg, x_train, x_test, y_train, y_test)

In [ ]:
k_folds = 5
n_depths_range = np.arange(5, 7, 1, dtype=np.int16)
n_estimators_range = np.arange(60, 150, 30, dtype=np.int16) # CV number of trees (estimators) 10, 20, ..., 100
min_leaf_samples_range = np.arange(3, 5, 1, dtype=np.int16)

best_veg = grid_search(x_train, y_train, n_depths_range, n_estimators_range, min_leaf_samples_range, k_folds, vegReg, score = 'neg_mean_squared_error')

In [ ]:
evaluate_reg(best_veg, x_train,x_test, y_train, y_test, calc_shap = False)

In [ ]:
# Try kfold again allowing whatever depth it wants, just trying number of trees
# Also, don't use vegReg, use a new regressor
k_folds = 5
n_depths_range = None
n_estimators_range = np.arange(100, 200, 30, dtype=np.int16)
min_leaf_samples_range = None

best_veg_2 = grid_search(x_train, y_train, n_depths_range, n_estimators_range, min_leaf_samples_range, k_folds, RandomForestRegressor(random_state=1), score = 'neg_mean_squared_error')

In [ ]:
evaluate_reg(best_veg_2, x_train,x_test, y_train, y_test, calc_shap = False)

Plotting

In [ ]:
vegpred_train = best_veg.predict(x_train)
vegpred_test = best_veg.predict(x_test)

fig, ax = plt.subplots(figsize=(6,6))

ax.scatter(y_train, vegpred_train, c='blue', s=4)
ax.scatter(y_test, vegpred_test, c='green', s=4)
ax.set_xlabel('Observed')
ax.set_ylabel('Predicted')
#ax.set_xlim(0, 10)
#ax.set_ylim(0, 10)
plt.show()

### World Data
Now we try again using the world data.

In [ ]:
# Use world data
selected_df = train_filtered.copy()
trainData_y, testData_y = train_test_split(selected_df, test_size=0.2, random_state=42)

#Y-values for train and test
y_train = trainData_y['VegC']
y_test  = testData_y['VegC']

print(y_train.shape)
print(y_test.shape)

#X-values for train and test
x_train = trainData_y.drop(['Pan_2007', 'Biome_obs', 'Lon', 'cellfraction', 'VegC', 'NPP'], axis=1)

x_test = testData_y.drop(['Pan_2007', 'Biome_obs', 'Lon', 'cellfraction', 'VegC', 'NPP'], axis=1)
print(f"Training set: {x_train.shape}")
print(f"Test set: {x_test.shape}")

In [ ]:
vegReg_world = RandomForestRegressor(random_state=42)

vegReg_world.fit(x_train, y_train)

In [ ]:
evaluate_reg(vegReg_world, x_train, x_test, y_train, y_test)

In [ ]:
vegpred_train = vegReg_world.predict(x_train)
vegpred_test = vegReg_world.predict(x_test)

fig, ax = plt.subplots(figsize=(6,5))

ax.scatter(y_train, vegpred_train, c='blue', s=4, label='Training data')
ax.scatter(y_test, vegpred_test, c='green', s=4, label='Test data')
ax.set_xlabel('Observed')
ax.set_ylabel('Predicted')
ax.set_title("Observed vs predicted VegC values")
#ax.set_xlim(0, 10)
ax.set_xlim((0, np.max(y_train)))
ax.set_ylim((0, np.max(y_train)))
ax.legend()
#ax.set_ylim(0, 10)
plt.show()

As seen above, the data from the world is reasonably well predicted. As in, it matches between training and test sets fairly well. The MSE is also noticably lower.

### Comparing to LPJGuess
Here we compare our predictions of VegC to those of LPJGuess. Since these are the values we used as our target variable, the MSE metric already compares them. However, we plot the two below to further experiment.

In [ ]:
# Plot our regresion predictions vs lpj guess (just y_test)
vegc_pred = vegReg_world.predict(x_test)

# Plot the values
#x_pts = np.arange(len(vegc_pred))
x_pts = y_test
plt.plot(x_pts, y_test, c='r', lw=2, label='LPJ VegC')
plt.scatter(x_pts, vegc_pred, label='Our predictions')
#plt.hist(biome_pred, bins=bin_num, alpha=0.5, label='Our predictions')
#plt.hist(biome_lpj_cmax, bins=bin_num, alpha=0.5, label='LPJ VegC')
plt.title("Comparing our VegC predictions to LPJ Guess")
plt.xlabel("LPJ VegC")
plt.ylabel("Predicted")
plt.xlim((0, np.max(y_test)))
plt.ylim((0, np.max(y_test)))
plt.legend()
plt.show()

## 5.2: NPP
Prediction of NPP using our features and the LPJ Guess output as the real value.

### Regional
Here we again use Americas, USA, and Canada for training and Russia for testing. This time looking at NPP.

In [ ]:
#regression NPP

# Extract values only from trainCountry and testCountry
train_region = ['Americas', 'USA', 'Canada']
test_region = ['Russia']

selected_df = train_filtered[train_filtered['Pan_2007'].isin(train_region+test_region)]

#We split train and test set according to countries as stated in 3.1 Data split in the PDF

#contains OBS which is our y-values
trainData_y = selected_df[selected_df['Pan_2007'].isin(train_region)]

testData_y = selected_df[selected_df['Pan_2007'].isin(test_region)]

print(len(trainData_y))
#Y-values for train and test

#WE NEED TO SELECT ONE BIOME, see legend of biomes file

selected_biome = 2 # boreal ever forest
#5 temp decid forest

#we want all biome categories

y_train = trainData_y['NPP']
y_test  = testData_y['NPP']

print(y_train.shape)
print(y_test.shape)

#X-values for train and test
x_train = trainData_y.drop(['Pan_2007', 'Biome_obs', 'Lon', 'cellfraction', 'VegC', 'NPP'], axis=1)

x_test = testData_y.drop(['Pan_2007', 'Biome_obs', 'Lon', 'cellfraction', 'VegC', 'NPP'], axis=1)

In [ ]:
nppReg = RandomForestRegressor(random_state=42)

nppReg.fit(x_train, y_train)

In [ ]:
mean_squared_error(y_test, nppReg.predict(x_test))

In [ ]:

k_folds = 5
n_depths_range = np.arange(5, 7, 1, dtype=np.int16)
n_estimators_range = np.arange(60, 150, 30, dtype=np.int16) # CV number of trees (estimators) 10, 20, ..., 100
min_leaf_samples_range = np.arange(3, 5, 1, dtype=np.int16)

best_npp = grid_search(x_train, y_train, n_depths_range, n_estimators_range, min_leaf_samples_range, k_folds, nppReg, score = 'neg_mean_squared_error')

In [ ]:
# Try kfold again allowing whatever depth it wants, just trying number of trees
# Also, don't use nppReg, use a new regressor
k_folds = 5
n_depths_range = None
n_estimators_range = np.arange(100, 200, 30, dtype=np.int16)
min_leaf_samples_range = None

best_npp_2 = grid_search(x_train, y_train, n_depths_range, n_estimators_range, min_leaf_samples_range, k_folds, RandomForestRegressor(random_state=1), score = 'neg_mean_squared_error')

In [ ]:
evaluate_reg(best_npp_2, x_train,x_test, y_train, y_test, calc_shap = False)

In [ ]:
npppred_train = best_npp.predict(x_train)
npppred_test = best_npp.predict(x_test)

fig, ax = plt.subplots(figsize=(6,6))

ax.scatter(y_train, npppred_train, c='blue', s=4)
ax.scatter(y_test, npppred_test, c='green', s=4)
ax.set_xlabel('Observed')
ax.set_ylabel('Predicted')
#ax.set_xlim(0, 10)
#ax.set_ylim(0, 10)
plt.show()

In [ ]:
evaluate_reg(best_npp, x_train,x_test, y_train, y_test, calc_shap = False)

### World Data
Now we try again using the world data.

In [ ]:
# Use world data
selected_df = train_filtered.copy()
trainData_y, testData_y = train_test_split(selected_df, test_size=0.2, random_state=42)

#Y-values for train and test
y_train = trainData_y['NPP']
y_test  = testData_y['NPP']

print(y_train.shape)
print(y_test.shape)

#X-values for train and test
x_train = trainData_y.drop(['Pan_2007', 'Biome_obs', 'Lon', 'cellfraction', 'VegC', 'NPP'], axis=1)

x_test = testData_y.drop(['Pan_2007', 'Biome_obs', 'Lon', 'cellfraction', 'VegC', 'NPP'], axis=1)
print(f"Training set: {x_train.shape}")
print(f"Test set: {x_test.shape}")

In [ ]:
nppReg_world = RandomForestRegressor(random_state=42)

nppReg_world.fit(x_train, y_train)

In [ ]:
evaluate_reg(nppReg_world, x_train, x_test, y_train, y_test)

In [ ]:
npppred_train = nppReg_world.predict(x_train)
npppred_test = nppReg_world.predict(x_test)

fig, ax = plt.subplots(figsize=(6,5))

ax.scatter(y_train, npppred_train, c='blue', s=4, label='Training data')
ax.scatter(y_test, npppred_test, c='green', s=4, label='Test data')
ax.set_xlabel('Observed')
ax.set_ylabel('Predicted')
ax.set_title("Observed vs predicted NPP values")
#ax.set_xlim(0, 10)
ax.set_xlim((0, np.max(y_train)))
ax.set_ylim((0, np.max(y_train)))
ax.legend()
#ax.set_ylim(0, 10)
plt.show()

As seen above, the data from the world is reasonably well predicted. As in, it matches between training and test sets fairly well. The MSE is also noticably lower than in the regional case.

### Comparing to LPJGuess
Here we compare our predictions of NPP to those of LPJGuess. Since these are the values we used as our target variable, the MSE metric already compares them. However, we plot the two below to further experiment.

In [ ]:
# Plot our regresion predictions vs lpj guess (just y_test)
npp_pred = nppReg_world.predict(x_test)

# Plot the values
x_pts = y_test
plt.plot(x_pts, y_test, c='r', lw=2, label='LPJ NPP')
plt.scatter(x_pts, npp_pred, label='Our predictions')
plt.title("Comparing our NPP predictions to LPJ Guess")
plt.xlabel("LPJ NPP")
plt.ylabel("Predicted")
plt.xlim((0, np.max(y_test)))
plt.ylim((0, np.max(y_test)))
plt.legend()
plt.show()